# 01 Model Serving Api

## 📚 Learning Objectives

By completing this notebook, you will:
- Expose a model via REST or similar API
- Implement request handling and basic validation

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

## Official Structure Reference

This notebook supports **Course 11, Unit 1** requirements from `DETAILED_UNIT_DESCRIPTIONS.md`.

---


# 01 Model Serving Api

## 📚 Learning Objectives | أهداف التعلم

This notebook demonstrates key concepts through hands-on examples.

By completing this notebook, you will:
- Understand the core concepts
- See practical implementations
- Be ready for exercises

## 🔗 Prerequisites | المتطلبات الأساسية

- ✅ Python 3.8+ installed
- ✅ Required libraries (see `requirements.txt`)
- ✅ Basic Python knowledge

---

## Code Example | مثال الكود

Run the code below to see the demonstration:


"""Unit 1 - Example 1: Model Serving with REST APIالوحدة 1 - مثال 1: تقديم النموذج باستخدام REST APIThis example demonstrates:1. Saving and loading a model2. Creating a REST API endpoint3. Serving predictions via API"""import joblibimport numpy as npfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.datasets import make_classificationprint("=" * 60)print("Example 1: Model Serving with REST API")print("مثال 1: تقديم النموذج باستخدام REST API")print("=" * 60)# 1. Train and Save Model# تدريب وحفظ النموذجprint("\n1. Training and Saving Model")print("تدريب وحفظ النموذج")print("-" * 60)# Create sample dataX, y = make_classification(n_samples=100, n_features=4, random_state=42)# Train modelmodel = RandomForestClassifier(n_estimators=10, random_state=42)model.fit(X, y)print("✓ Model trained successfully")print("✓ تم تدريب النموذج بنجاح")# Save modelmodel_filename = 'trained_model.pkl'joblib.dump(model, model_filename)print(f"✓ Model saved to {model_filename}")print(f"✓ تم حفظ النموذج في {model_filename}")# Load modelloaded_model = joblib.load(model_filename)print("✓ Model loaded successfully")print("✓ تم تحميل النموذج بنجاح")# 2. API Structure (using Flask)# هيكل API (باستخدام Flask)print("\n" + "=" * 60)print("2. REST API Structure")print("هيكل REST API")print("=" * 60)api_code = """from flask import Flask, request, jsonifyimport joblibimport numpy as npapp = Flask(__name__)# Load model at startupmodel = joblib.load('trained_model.pkl')@app.route('/predict', methods=['POST'])def predict():    # Get data from request    data = request.json    features = np.array(data['features']).reshape(1, -1)        # Make prediction    prediction = model.predict(features)[0]    probability = model.predict_proba(features)[0].tolist()        return jsonify({        'prediction': int(prediction), 'probability': probability    })@app.route('/health', methods=['GET'])def health():    return jsonify({'status': 'healthy'})if __name__ == '__main__':    app.run(host='0.0.0.0', port=5000, debug=True)"""print(api_code)# 3. Example API Request# مثال على طلب APIprint("\n" + "=" * 60)print("3. Example API Request")print("مثال على طلب API")print("=" * 60)example_request = """POST http://localhost:5000/predictContent-Type: application/json{    "features": [0.5, 0.3, 0.8, 0.2]}"""print(example_request)example_response = """Response:{    "prediction": 1, "probability": [0.2, 0.8]}"""print(example_response)# Test the loaded modelprint("\n" + "=" * 60)print("4. Testing Loaded Model")print("اختبار النموذج المحمّل")print("=" * 60)test_sample = np.array([[0.5, 0.3, 0.8, 0.2]])prediction = loaded_model.predict(test_sample)probability = loaded_model.predict_proba(test_sample)print(f"\nTest sample: {test_sample[0]}")print(f"Prediction: {prediction[0]}")print(f"Probability: {probability[0]}")print("\n" + "=" * 60)print("Example completed successfully!")print("تم إكمال المثال بنجاح!")print("=" * 60)print("\nNote: Install Flask to run API:")print("pip install flask")

## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [ ]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

---

## ✅ Summary | الملخص

Great job completing this example!

**What you learned:**
- Core concepts demonstrated in the code
- Practical implementation details

**Next steps:**
- Complete the exercises in `exercises/` folder
- Review the quiz materials
- Proceed to the next example

---

**💡 Tip:** If you see errors, make sure:
- All libraries are installed: `pip install -r requirements.txt`
- You're using Python 3.8+
- Cells are executed in order


## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.